# 03 · 少看历史会丢掉什么：四种读取方式在真实问题上的质量对照

**这是一份只读实验课：** 它加载 `reports/` 下已保存的 G2-lite 结果（基线可用性、四方法对照、覆盖率、代表案例的真实选择位置、反事实改写、成本），展示与那些结果同一版本的正式源码，并用一个随机初始化的 2 层小 Qwen3（真实模块代码，CPU float64）重放 B / C / D0 三种选择器的契约。它不加载 0.6B 模型、不跑 GPU、不训练。重跑的入口是 `bench/g2lite_quality.py`（三遍：A 基线 → 可用格子上的四方法对照 → 反事实改写）与 `scripts/g2lite_report.py`，不由本页隐式触发。

G1b 回答了速度一侧：同模型同设备下，未训练索引读取的整步收益不足 10%，速度主线收口。本页回答质量一侧的一个缩减问题：**当模型只能读一小部分历史时，它丢掉了什么？** 四种读取方式在同一前缀、同一问题、同一名额下作答：

| 方法 | 读什么 | 名额给谁 |
|---|---|---|
| A 完整读取 | 全部合法位置 | — |
| B 近期窗口 | F ∪ 最近的 k 个候选位置 | 最近的位置 |
| C 均匀随机 | F ∪ 候选域内均匀抽取的 k 个位置（固定种子，每层每组独立） | 随机 |
| D0 未训练索引 | F ∪ 随机初始化 GroupedIndexer 打分的 top-k | 由内容决定但权重未训练 |

F = 首 4 ∪ 近 64；B、C、D0 的实际读取数都是 68 + k0。**C 不叫"随机筛选"，D0 也不叫**：C 的选择与内容无关，D0 的选择由内容决定但没有学过。每个案例按固定结构：资料与问题 → 运行前预测 → 四种方法实际回答 → 各自选择的位置 → 改动一处资料再检查 → 哪些解释得到支持、哪些仍不确定。模型成绩来自 2026-10-17 保存的 Mac M4 Max / MPS 运行；本次报告修订与 CPU 契约不产生新的模型成绩；**48 份是探索样本，不做区间统计，不支持"损失小于 2 个百分点"或其反面；D0 未训练，B 的名单按窗口几何确定，正确性另测。**


**2026-10-18 审查修订。** 主表准确率保持不变；修正改写计数、案例邻域和预算口径。旧 C 改写同时改变了资料和随机名单，保留为混杂观察。当前配置 v3 的稳定父资料种子规则尚未做模型运行；本页结果仍来自 v2。学习重点是分清观察、充分条件与必要条件。

## 0. 定位仓库、加载已保存结果、核对源码版本

结果与代码必须锁在同一版本：下面把当前源码的哈希与三遍 run 的 manifest 记录的哈希逐个对照，并核对模型 / tokenizer revision、配置版本与数据 digest。

In [ ]:
from pathlib import Path
import hashlib, inspect, importlib.util, json, sys, textwrap
from IPython.display import display, Markdown
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".attention_lab_bootstrap").is_file()), None)
assert root is not None, "请从 attention-lab 项目目录或 notebooks 子目录打开本页"
sys.path.insert(0, str(root / "src"))

def load(rel):
    p = root / rel
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

summary = load("reports/g2lite_summary_mac_m4max.json")      # 表格、案例、改写、成本（精简）
answers = load("reports/g2lite_answers_mac_m4max.json")      # 逐题回答与解析结果
manifests = load("reports/g2lite_manifest_mac_m4max.json")   # 三遍 run 的 manifest
cfg = load("reports/provenance/g2lite_pr5_review/mac_g2lite_at_formal_runs.json")  # 运行时 v2
build = load("reports/g2lite_report_build.json")
review_sources = load("reports/provenance/g2lite_pr5_review_manifest.json")
def load_module(name, relative_path):
    spec = importlib.util.spec_from_file_location(name, root / relative_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
reporting = load_module("lesson03_reporting", "scripts/g2lite_report.py")
if summary is None or answers is None or manifests is None:
    print("未执行：reports/ 下没有 G2-lite 结果。入口：python bench/g2lite_quality.py --methods A --tag pass1-A 等三遍，再 python scripts/g2lite_report.py")
else:
    print("基线 run:", summary["baseline_run"], "| 对照 run:", summary["comparison_run"], "| 改写 run:", summary["rewrite_runs"])
    print("配置版本:", summary["config_version"], "| ρ:", summary["rho"], "| 可用格子:", summary["usable_cells"])
    for m in manifests["runs"]:
        print(f"{m['run_id']:<40s} commit {m['git']['commit'][:10]} dirty {m['git']['dirty_diff_sha256']} model {m['model_revision'][:10]} tokenizer {m['tokenizer_revision'][:10]} data {m['data_sha256'][:12]} config v{m['config_version']} HF_HUB_OFFLINE={m['hf_hub_offline']}")


In [ ]:
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
if manifests:
    snapshots = {item["source"]: item["snapshot"] for item in review_sources["snapshots"]}
    files = ["src/attention_lab/decode.py", "src/attention_lab/qwen_wrap.py", "src/attention_lab/g2lite_data.py", "src/attention_lab/g2lite_eval.py", "bench/g2lite_quality.py"]
    for relative_path in files:
        as_run = root / snapshots.get(relative_path, relative_path)
        recorded_hash = sha(as_run)
        assert all(m["source_hashes"][relative_path] == recorded_hash for m in manifests["runs"])
        print(relative_path, "运行时源码核对通过", "当前文件相同" if sha(root / relative_path) == recorded_hash else "当前已修复；旧版本由快照保留")
    for relative_path, expected in build["report_source_hashes"].items():
        assert sha(root / relative_path) == expected
    print("报告生成提交:", build["source_commit"], "| 新模型运行:", build["new_model_runs"])
    print("旧数据版本:", build["model_run_config_versions"], "| 当前修复配置:", load("configs/mac_g2lite.json")["config_version"], "NOT_RUN")

**读法。** 四个正式 run 使用 `af62df4` 的源码、配置 v2，模型与 tokenizer revision 同为 `c1899de…`。基线、可用格子对照和两组改写的资料集合不同，数据 digest 各自不同；不能把“相同生成规则”写成“相同数据摘要”。

选择器和 Qwen 封装未改，本页的 CPU 演示调用它们的原实现。运行器和汇总逻辑已有修复：运行时版本由来源快照保存，今日报告另记生成代码提交与 SHA-256。旧 run/manifest 不改，旧 C 的混杂不会因为今天修了代码而消失。

## 1. 四种读取方式的正式代码与契约

**运行前的预测**（`reports/cards/G2lite_mac_m4max.md`）：B、C、D0 在同一 ρ 下读取数相同、都包含 F；全选时三者退化为 A；B 的额外行恰为最近的 k 个候选；C 的抽取由种子决定且回退重放不变；B、C 不打分。下面并排打印三个名单生成函数与共用的取行核，再在小模型上逐条验证。

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import torch
from attention_lab import decode, qwen_wrap
for fn in (decode.select_rows_kernel, decode.select_rows_recent, decode.select_rows_uniform, decode.gathered_decode_step_kernel):
    print(inspect.getsource(fn))


In [ ]:
# 封装里的分派：同一个解码分支，只有名单来源不同
src = inspect.getsource(qwen_wrap.LabAttention.forward)
i = src.index('if st.mode == "sparse":'); j = src.index("else:   # prefill")
print(textwrap.dedent(src[i:j]))


In [ ]:
# 小模型契约重放（CPU float64，2 层随机 Qwen3）：同 F 同名额同读取数；B = 最近 k 个候选；C 由种子决定；全选退化为 A；污染未选行不改变 B / C
from attention_lab.qwen_wrap import tiny_qwen3, wrap_model, unwrap_model, make_state, LabRunner
from attention_lab.selection import FIXED_PREFIX, FIXED_RECENT, fixed_budget
torch.manual_seed(0)
m = tiny_qwen3(); ids = torch.randint(0, 256, (1, 130), generator=torch.Generator().manual_seed(31))
holder = wrap_model(m); PRE = 100
def run_sel(selector, k0=None, seed=3):
    st = make_state(m, 160, "sparse", selector=selector, uniform_seed=seed); run = LabRunner(m, holder, st)
    run.prefill(ids[:, :PRE], 37); st.begin_request(None if k0 else 1/8, k0=k0); st.record_selection = True
    out = run.step(ids[:, PRE:PRE+1]); return st, run, out
with torch.inference_mode():
    S = {name: run_sel(sel) for name, sel in (("B", "recent"), ("C", "uniform"), ("D0", "index"))}
    legal = PRE + 1; F = set(range(FIXED_PREFIX)) | set(range(legal - FIXED_RECENT, legal)); lo, hi = FIXED_PREFIX, legal - FIXED_RECENT
    print(f"合法长度 {legal}，k0 = {fixed_budget(legal, 1/8)}，候选域 [{lo}, {hi})，F 有 {len(F)} 个位置")
    for name, (st, run, out) in S.items():
        pos = (st.last_rows[0].view(st.cache.groups, -1) - st.cache.offsets.view(-1, 1))[0].tolist()
        extra = sorted(set(pos) - F)
        print(f"{name:<3s} 读取数 {len(pos)} = {st.plan.selected_count}  含 F: {F.issubset(pos)}  额外 {len(extra)} 个候选: {extra}")
    # 全选退化
    stA = make_state(m, 160, "full"); runA = LabRunner(m, holder, stA); runA.prefill(ids[:, :PRE], 37); outA = runA.step(ids[:, PRE:PRE+1])
    for name, sel in (("B", "recent"), ("C", "uniform"), ("D0", "index")):
        st, run, out = run_sel(sel, k0=160)
        print(f"全选 {name}: 读取数 {st.plan.selected_count} = 合法长度 {st.plan.legal_length}, 对 A 的 max|Δ| = {float((out - outA).abs().max()):.2e}")
    # 污染未选行：B / C 逐位不变，A 坏掉
    for name, sel in (("B", "recent"), ("C", "uniform")):
        st, run, clean = run_sel(sel)
        stA2 = make_state(m, 160, "full"); runA2 = LabRunner(m, holder, stA2); runA2.prefill(ids[:, :PRE], 37); runA2.step(ids[:, PRE:PRE+1])
        for l in range(st.cache.layers):
            pos = (st.last_rows[l].view(st.cache.groups, -1) - st.cache.offsets.view(-1, 1))
            for g in range(st.cache.groups):
                keep = torch.zeros(160, dtype=torch.bool); keep[pos[g]] = True
                st.cache.k[l][0, g, ~keep] = float("nan"); st.cache.v[l][0, g, ~keep] = float("nan")
                stA2.cache.k[l][0, g, ~keep] = float("nan"); stA2.cache.v[l][0, g, ~keep] = float("nan")
        run.rollback(PRE); runA2.rollback(PRE)
        again = run.step(ids[:, PRE:PRE+1])
        print(f"污染未选行后 {name}: 逐位相同 = {torch.equal(again, clean)}；完整读取 A 出现 NaN = {not torch.isfinite(runA2.step(ids[:, PRE:PRE+1])).all().item()}")
    unwrap_model(m)


**解释。** B、C、D0 在同题同档使用相同 F 和名额，区别是每一步选择的位置以及随后各自形成的状态。污染测试固定已经算好的缓存，只检查本次读取是否触及名单之外的行；它不证明“重建资料后，名单外的原文不能影响名单内的深层 K/V”。后面的改写必须保留这一区别。

## 2. 资料、问题与协议

**协议 `document_dense_then_sparse`。** 资料前缀（聊天模板头 + 表头 + 记录与填充）用完整读取处理一次并建索引表，问题不在前缀里；每个请求克隆这份前缀缓存，从问题第一个 token 起逐 token 用自己的读取方式（问题、模板尾部、生成的每个 token 都是一步解码）；贪心，≤ 64 新 token，EOS 停止；资料改写后重建前缀。下面展示生成器的布局函数与一份实际资料。

In [ ]:
from attention_lab import g2lite_data
as_run_evaluation = load_module("lesson03_as_run_evaluation", "reports/provenance/g2lite_pr5_review/g2lite_eval_at_formal_runs.py")
print(inspect.getsource(g2lite_data.layout_document)[:2600])
print(inspect.getsource(as_run_evaluation.score_answer))

In [ ]:
if summary:
    d = summary["docs"]["T1-2K-00"]
    print("doc_id T1-2K-00 ·", d["task"], d["length_tag"], "· 目标桶", d["buckets"], "· 记录数", d["n_records"], "· 前缀 token", d["prefix_token_count"], "· 问题+模板尾 token", d["suffix_token_count"])
    print("问题:", d["question"], "| 目标值:", d["targets"], "| 目标距前缀末尾 token:", d["target_distance_from_prefix_end"], "| 倒数第几条:", d["target_records_from_end"])
    print()
    print("几何（全部正式资料）：")
    rows = sorted((v for v in summary["docs"].values() if v["variant"] == "base"), key=lambda v: (v["task"], v["length_tag"]))
    print(f"{'doc':<10} {'桶':<18} {'前缀':>6} {'尾部':>4} {'目标距末尾 token':>18} {'倒数第几条':>10}")
    for v in rows:
        print(f"{v['task']+'-'+v['length_tag']+'-'+v['question'][3:7]:<10} {'/'.join(v['buckets']):<18} {v['prefix_token_count']:>6} {v['suffix_token_count']:>4} {str(v['target_distance_from_prefix_end']):>18} {str(v['target_records_from_end']):>10}")


**读法。** 每份资料 16 条记录穿插在无数字的中文填充句之间；问题与模板尾部为 35（T1）或 54（T2）token。ρ 在请求起始合法长度 N0=P+1 上决定固定 k0，实际前缀略有长短，因此相同 ρ 下不同资料的 k0 可以略有不同；同一资料的 B/C/D0 则相同。

预测第一个答案 token 时，问题已处理完，合法长度 N=P+Q，候选域大小 M=N−|F|。需要分别看 **ρ、k/M、(F+k)/N**；下面按逐题记录展示范围。B 此时覆盖的资料尾部长度为 64+k0−Q。记录是否落入窗口是可检查的几何关系，不预先保证模型答对。

## 3. 基线可用性（第一遍：A 在全部格子）

**运行前的预测**（卡片原文，写于 v1 密集表设计）：T1-2K 10–12/12，T2-2K 7–11/12，T1-8K 6–11/12，T2-8K 3–9/12。**校准后的修订预测**（卡片附录，写于正式运行前）：T1 两档可用（2K 10–12，8K 9–12），T2 两档不可用（2K 3–7，8K 0–3）。门槛 80% 是小样本可用性检查。

In [ ]:
if summary:
    print(f"{'格子':<8} {'n':>3} {'对':>3} {'率':>6} {'答成编号':>8} {'任意顺序含全部目标':>18} {'格式错误':>8} {'截断':>5}  按目标位置")
    base_rows = [r for r in answers if r["run_id"] == summary["baseline_run"]]
    for c, t in summary["baseline_methods"].items():
        a = t["A"]; rows_c = [r for r in base_rows if f"{r['task']}/{r['length_tag']}" == c]
        print(f"{c:<8} {a['n']:>3} {a['hits']:>3} {a['rate']*100:>5.0f}% {sum(r['answered_with_ids'] for r in rows_c):>8} {a['any_order']*100:>17.0f}% {a['format_errors']:>8} {a['truncated']:>5}  " + "; ".join(f"{b} {v*100:.0f}%" for b, v in a["by_bucket"].items()) + f"   可用: {summary['gate'][c]['usable']}")
    print()
    for r in base_rows:
        if not r["exact_match"]:
            print(f"  ✗ {r['doc_id']:<10} 桶 {'/'.join(r['buckets']):<18} 目标 {r['targets']} 回答 {r['predicted']} 文本 {r['text'][:40]!r}")


**观察。** T1 两档基线达到本批探索样本的可用性门槛，T2 两档未达到。A 的 T1 错误发生在 8K，回答了其他记录的值，没有编号回声。T2 有编号回声、一对一错或全错等形态；8K 还出现“编号:值”配对格式。冻结评分取最前面的一个或两个独立四位数，不能与“整段包含正确内容”混同。

**边界。** `baseline_too_weak` 仅指当前提示、生成和评分协议下没有过门槛，不是“模型没有双证据读取能力”。本轮未改评分规则，T2 不进入稀疏对照；80% 也不是总体能力的区间估计。

## 4. 四方法质量对照（第二遍，只在可用格子）

**运行前的预测。** B 由几何决定：T1 在 2K · ρ = 1/32 只对 back_near 的 2/12，其余三个设置对 back_near + back_far 的 4/12；T2 0–1/12。C 接近零（1/32 下 0/12，1/8 下 0–2/12）。D0 不优于 C（每格 0–2/12）。

In [ ]:
if summary and summary["comparison_methods"]:
    print("A 重跑一致性:", summary["a_consistency_vs_baseline"])
    print(f"{'格子':<8} {'方法':<10} {'k0':>5} {'读取':>5} {'对/n':>6} {'答成编号':>8} {'任意顺序':>8} {'同对/A对它错/A错它对/同错':>26}  按目标位置")
    comp_rows = [r for r in answers if r["run_id"] == summary["comparison_run"] and r["variant"] == "base"]
    for c, tbl in summary["comparison_methods"].items():
        for mk, t in tbl.items():
            rows_c = [r for r in comp_rows if f"{r['task']}/{r['length_tag']}" == c and (r["method"] if r["rho"] is None else f"{r['method']}@{r['rho']:g}") == mk]
            pr = (summary["paired"].get(c) or {}).get(mk)
            p2 = f"{pr['counts']['both_right']}/{pr['counts']['A_right_other_wrong']}/{pr['counts']['A_wrong_other_right']}/{pr['counts']['both_wrong']}" if pr else "—"
            print(f"{c:<8} {mk:<10} {reporting.value_range(t['k0']):>5} {(reporting.value_range(t['selected']) if t['selected'] else '全部'):>5} {t['hits']:>2}/{t['n']:<3} {sum(r['answered_with_ids'] for r in rows_c):>8} {t['any_order']*100:>7.0f}% {p2:>26}  " + "; ".join(f"{b} {v*100:.0f}%" for b, v in t["by_bucket"].items()))


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
if summary and summary["comparison_methods"]:
    cells_ = list(summary["comparison_methods"].keys())
    fig, axes = plt.subplots(1, len(cells_), figsize=(5 * len(cells_), 3.6), squeeze=False)
    for ax, c in zip(axes[0], cells_):
        tbl = summary["comparison_methods"][c]; names = list(tbl.keys()); vals = [tbl[k]["rate"] * 100 for k in names]
        ax.bar(range(len(names)), vals, color=["#444" if k == "A" else ("#1f77b4" if k.startswith("B") else "#ff7f0e" if k.startswith("C") else "#2ca02c") for k in names])
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_ylim(0, 100); ax.set_title(f"{c}: exact match (%) · n={tbl['A']['n']}"); ax.set_ylabel("exact match %")
    plt.tight_layout(); plt.show()


In [ ]:
# 覆盖率：每步每层每组读到的被查询记录 token 占比；与 k/M 的期望并排
if summary and summary["comparison_methods"]:
    import statistics
    comp_rows = [r for r in answers if r["run_id"] == summary["comparison_run"] and r["variant"] == "base" and r["method"] != "A"]
    print(f"{'长度':<4} {'方法':<10} {'k/M 期望':>9} {'问题阶段':>9} {'生成阶段':>9} {'首生成步 均值/最低层/最高层':>28}")
    for lt in sorted({r["length_tag"] for r in comp_rows}):
        for mk in sorted({f"{r['method']}@{r['rho']:g}" for r in comp_rows}):
            rs = [r for r in comp_rows if r["length_tag"] == lt and f"{r['method']}@{r['rho']:g}" == mk]
            if not rs: continue
            budget = summary["read_budgets"][f"T1/{lt}"][mk]
            fl = [r["coverage_first_gen_step_per_layer"] for r in rs]
            per_layer = [statistics.mean(x[l] for x in fl) for l in range(len(fl[0]))]
            print(f"{lt:<4} {mk:<10} {budget["candidate_fraction_mean"]*100:>8.1f}% {statistics.mean(r['coverage_question_steps'] for r in rs)*100:>8.1f}% {statistics.mean(r['coverage_generation_steps'] for r in rs if r['coverage_generation_steps'] is not None)*100:>8.1f}% {statistics.mean(r['coverage_first_gen_step_mean'] for r in rs)*100:>9.1f}% / {min(per_layer)*100:.1f}% / {max(per_layer)*100:.1f}%")

    lines = ["| 格子 | ρ | 实际 k0 | 总读取数 | 首生成步 N | k/M | 总读取比例 |", "|---|---:|---|---|---|---|---|"]
    for cell, table in summary["read_budgets"].items():
        for method, budget in table.items():
            if not method.startswith("B@"):
                continue
            vr = reporting.value_range
            lines.append(f"| {cell} | {budget['rho']:g} | {vr(budget['k0'])} | {vr(budget['selected'])} | {vr(budget['legal'])} | {vr(budget['candidate_fraction'], percent=True)} | {vr(budget['total_read_fraction'], percent=True)} |")
    display(Markdown("\n".join(lines)))


**观察。** 主表和配对结果与旧记录一致：C/D0 在四格均未答对；B 窗口内 9/14、窗口外 0/34，且有增加名额后由对变错的案例。多数失败答成问题里的编号。C 的目标位置覆盖接近对应步骤的 k/M；D0 的覆盖较高，但分数仍为零。

**解释的边界。** 这里同时观察到了低目标位置覆盖和零分，不能仅凭这个指标确定两者的充分或必要关系。覆盖指标计算整条记录，而非仅四位答案；跨层、跨位置的信息传递也没有被分离。“没直接选到目标行”和“没有任何目标信息”不是同一件事。

D0 未训练，其零分不代表训练后索引的上限。B 的结果限定于当前 F、预算和样本。计时含覆盖率记账与同步，不能作为新的速度基准。

## 5. 代表案例

预登记的选法：每个可用格子、每一类（A 对 B 错；A 对 C 或 D0 错；四种都对）按资料序号取最多两份；存在才展示。每个案例按固定结构：资料与问题 → 运行前预测 → 四种方法实际回答 → 各自选择的位置 → 改动一处资料再检查 → 哪些解释得到支持、哪些仍不确定。下面的代码按类别打印案例；解释写在其后的文字格里。

In [ ]:
def show_case(doc_id):
    d = summary["case_details"][doc_id]; doc = d["doc"]
    print("=" * 110)
    print(f"资料 {doc_id} · {doc['task']} {doc['length_tag']} · 目标桶 {doc['buckets']} · 倒数第 {doc['target_records_from_end']} 条 · 距前缀末尾 {doc['target_distance_from_prefix_end']} token · 前缀 {doc['prefix_token_count']} token")
    print("问题:", doc["question"])
    print("目标值:", doc["targets"])
    neighborhood = ["| 原记录序号（从 0 计数） | 编号 | 值 | 是否目标 |", "|---:|---|---|---|"]
    for record_index, (record_id, value) in d["records_around_targets"].items():
        neighborhood.append(f"| {record_index} | {record_id} | {value} | {'目标' if int(record_index) in doc['target_indices'] else '邻近'} |")
    display(Markdown("\n".join(neighborhood)))
    print("资料末尾六条 (编号, 值):", d["last_records"])
    print(f"{'方法':<10} {'读取':>5} {'回答':<34} {'判定':<8} {'目标覆盖 首生成步 均值':>22} {'层组全覆盖比例':>14}")
    for mk, an in d["answers"].items():
        cov = an["coverage"]
        print(f"{mk:<10} {str(an['selected_count'] or '全部'):>5} {an['text'][:32]!r:<34} {'✓' if an['exact'] else '✗'}{'(答成编号)' if an.get('answered_with_ids') else '':<8} {('全部' if mk == 'A' else f'{cov.get('first_gen_step_mean', 0)*100:.1f}%'):>22} {('—' if mk == 'A' else f'{cov.get('fraction_of_layer_groups_with_full_coverage_first_gen_step', 0)*100:.0f}%'):>14}")
    for mk, an in d["answers"].items():
        pos = an.get("positions")
        if not pos: continue
        ex = pos["example_rows"]
        print(f"  {mk} 第一个生成 token 那一步的名单（第 0 组）：" + "; ".join(f"{k}: {v['n_rows']} 行, {v['n_ranges']} 段, 前几段 {v['ranges'][:6]}" for k, v in ex.items()))
    if d["rewrites"]:
        print("  旧记录改写（C 的随机名单也变化）：")
        for r in d["rewrites"]:
            print(f"    {r['variant']} → {r['method']}{'@'+format(r['rho'],'g') if r['rho'] is not None else ''}: 答 {r['predicted']}，目标 {r['targets']}，{'✓' if r['exact'] else '✗'}")

def plot_case_coverage(doc_id):
    d = summary["case_details"][doc_id]
    fig, ax = plt.subplots(figsize=(8, 3))
    for mk, an in d["answers"].items():
        if mk == "A" or not an["coverage"].get("first_gen_step_per_layer"): continue
        ax.plot(an["coverage"]["first_gen_step_per_layer"], marker="o", ms=3, label=mk)
    ax.set_xlabel("layer"); ax.set_ylabel("fraction of target positions read"); ax.set_ylim(-0.02, 1.02); ax.set_title(f"{doc_id}: target coverage per layer at the first generated token"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

if summary:
    cats = {}
    for c in summary["cases"]:
        cats.setdefault(c["category"], []).append(c)
    print({k: [x["doc_id"] for x in v] for k, v in cats.items()})


### 5.1 A 对 B 错

**运行前的预测。** 大量存在（front / middle 目标）；关键记录在窗口外；把记录移近后 B 变对的比例 ≥ 3/4。

In [ ]:
if summary:
    for c in [x for x in summary["cases"] if x["category"] == "A_right_B_wrong"]:
        show_case(c["doc_id"]); plot_case_coverage(c["doc_id"])
    if not [x for x in summary["cases"] if x["category"] == "A_right_B_wrong"]:
        print("本类案例不存在（存在才展示）")


**哪些解释得到支持。** 这四份前段目标在 B 窗口外，A 答对而 B 答错。移近后的变化支持位置选择与表现有关，但不是保证：同一资料的不同预算仍可给出不同结果。窗口名单严格是 F ∪ 最近 k0 个候选，不能再把首 4 重复加入总读取数。

**反例与边界。** `T1-2K-00，B，ρ=1/8` 的目标仍在窗口外，value 改写却使回答由 8340 变为 1563，两次都错。第 6 节从逐题记录分别统计回答是否变化、正误是否变化、是否变成编号。“仍错”不代表“纹丝不动”。完整 Prefill 后的间接信息路径尚未定位，不能断言这条记录与输出无关。

### 5.2 A 对 C 或 D0 错

**运行前的预测。** 几乎处处存在；被查询记录的 token 在绝大多数层组里没有被选中，覆盖率 ≈ k/M。

In [ ]:
if summary:
    for c in [x for x in summary["cases"] if x["category"] == "A_right_C_or_D0_wrong"]:
        show_case(c["doc_id"]); plot_case_coverage(c["doc_id"])
    if not [x for x in summary["cases"] if x["category"] == "A_right_C_or_D0_wrong"]:
        print("本类案例不存在（存在才展示）")


**哪些解释得到支持。** C 和 D0 的选择位置分布不同，目标位置覆盖和成簇程度也不同，但本批请求都未答对。此处的覆盖率是整条记录位置被选中的比例，不是注意力权重，不等于四个值 token 的覆盖率。

**必要条件与充分条件要分开。** 若某层组读齐四个值 token 却仍错，它只说明“读齐”不足以保证答对；不能据此推出“不读齐也能答对”。原实验没有证明“必须同层组读齐”这个必要条件，因此应撤回结构性下界的解释。低覆盖与零分同时出现的观察保留；具体失败机制尚未分离。

### 5.3 四种方法都对

**运行前的预测。** 不存在或 ≤ 2 份；若出现，C / D0 在若干层里恰好选中了值的数字 token，而不是靠 F。

In [ ]:
if summary:
    for c in [x for x in summary["cases"] if x["category"] == "all_four_right"]:
        show_case(c["doc_id"]); plot_case_coverage(c["doc_id"])
    if not [x for x in summary["cases"] if x["category"] == "all_four_right"]:
        print("本类案例不存在（存在才展示）")


**本类不存在**：C 与 D0 没有答对任何一份资料，所以没有「四种方法都对」的样本。卡片预测的是「不存在或 ≤ 2 份」，成立。

## 6. 资料改写：先核对比较是否只改变一个条件

**运行前的预测保留。** A 改目标值应跟着改、改无关值应不变；B 窗口外改值应不变；C/D0 对无关改写的不变率有预期。完整预测原文保存在卡片，下面用旧记录逐项检查，不能把预测直接写成结论。

**C 的旧记录有混杂。** 原资料用随机名单 U，改后资料用另一名单 U′，因此同时改变了资料与随机抽样。新代码从 `parent_doc_id or doc_id` 派生种子，并通过 CPU 测试核对同候选域、k、请求计数和步号下的实际原位置名单。新规则是未运行的 v3；旧数据仍是 v2。D0 的参数与规则固定，但它随内容改变名单的响应应当保留。

In [ ]:
# 今日代码修复（v3，未运行新模型），不替换旧 v2 的回答和种子
control = load_module("lesson03_control", "src/attention_lab/g2lite_control.py")
print(inspect.getsource(control.uniform_seed_for))
current_config = load("configs/mac_g2lite.json")
base_id = "T1-2K-00"
for variant in ("base", "value", "irrelevant"):
    doc = {"doc_id": base_id if variant == "base" else base_id + "-" + variant,
           "parent_doc_id": None if variant == "base" else base_id}
    print(variant, control.uniform_seed_for(current_config, doc))
print("种子相同只是第一步。tests/test_g2lite.py 还在相同候选域、k、请求计数、步号下核对逐层逐组的实际原位置名单。D0 不使用这条外部随机种子规则。")

In [ ]:
if summary and summary["rewrites"]:
    display(Markdown(reporting.render_rewrite_observations(summary["rewrite_observations"])))
    print("回答改变但两次都错的 value 请求：")
    for r in summary["rewrites"]:
        if r["variant"] == "value" and r["answer_changed"] and r["both_wrong"]:
            print(r["parent_doc_id"], r["method"], r["rho"], r["base_predicted"], "→", r["rewrite_predicted"], "| 改后编号回声:", r["rewrite_answered_with_ids"])
    print("其余逐题改写见 reports/g2lite_summary_mac_m4max.json 的 rewrites；原始回答和评分在 answers 文件中保持不变。")

**观察。** 上表把“回答不变”“正误改变”“两次都错”“改后答编号”分开计算，没有用一个指标替代另一个。A 的改值／无关改写结果支持本批请求依赖资料；旧 C 的配对结果因外部随机条件不同不能判断固定随机读取下的单变量效应。D0 仍按完整的内容响应规则观察。

**可能的信息通路（机制假设，尚未定位具体层与位置）：**

```text
前方记录被改动
    ↓ 资料 Prefill 使用完整注意力
后方位置的深层表示与 K/V 可能变化
    ↓ 问题阶段继续读取这些位置
问题表示或最终回答可能变化
```

窗口外改值仍改变回答的例外，足以否定“窗口外必不变”。它没有证明上图某条具体通路已经被定位，也没有证明答案不变时完全不存在内部变化。本轮仅修正解释，不追加跨层干预。

## 7. 实际运行成本

In [ ]:
if summary:
    print("设备时间 s:", {k: (round(v) if isinstance(v, (int, float)) else v) for k, v in summary["device_seconds"].items()})
    for name, tm in (("baseline", summary["timing"]["baseline"]), ("comparison", summary["timing"]["comparison"]), *[(f"rewrites[{i}]", t) for i, t in enumerate(summary["timing"]["rewrites"])]):
        if tm:
            print(name, "| prefill ms 中位:", tm["prefill_ms_median_by_length"], "| 每步 ms 中位:", {k: round(v, 1) for k, v in tm["ms_per_step_median"].items()})


**读法。** 正式三遍 313 s，加冒烟与校准 654 s，现金 0，分配内存峰值 3.4 GB。每步耗时：A 11–13 ms，B 与 A 相当，D0 约 18 ms（G1b 测过的小核链），C 20–36 ms——C 的额外成本是 CPU 上 numpy 的均匀抽样与 top-k（每层每步一次，8K 候选域更大），不是 GPU 属性，也不是本任务的测量对象。

## 8. 预测对照与结论

| 项 | 预测 | 实测 | 判定 |
|---|---|---|---|
| 基线：T1 两档可用，T2 两档不可用 | 2K 10–12 / 8K 9–12；T2 3–7 / 0–3 | 12 / 10；8 / 1 | 成立（T2-2K 略高于区间，仍 < 80%） |
| B 窗口外全错 | 是 | 0/34 | 成立 |
| B 窗口内全对（T1 2/12、4/12；4/12、4/12） | 是 | 2、2；2、3（窗口内 9/14） | **失效**：窗口内命中率 64%，名额变大反而变错 |
| B 错时答窗口内近期记录的值或编造数 | 是 | 33/34 答编号 | 部分成立 |
| 移近后 B 变对 ≥ 3/4 | 是 | 按资料 3/4；按设置 4/8 | 资料口径成立 |
| C 接近零；覆盖率 = k/M | 是 | 0/12 ×4；偏差 ≤ 0.1 pp | 成立 |
| C 错误以编造数为主 | 是 | 47/48 编号回声 | 失效 |
| D0 不优于 C；覆盖率 ≈ k/M | 六成 | 0/12 ×4；覆盖率 1.2–2.2 × k/M | 成绩成立；覆盖率均值失效 |
| 四种都对不存在或 ≤ 2 | 是 | 不存在 | 成立 |
| 反事实签名 | 原预测见卡片 | 逐题动态计数见第 6 节 | A 在本批支持；B“窗口外必不变”被反例否定；旧 C 因混杂不能判断 |
| 成本 ≤ 40 min，峰值 < 8 GB | 是 | 11 min；3.4 GB | 成立；C / D0 每步成本高于预测 |

**结论（限 Qwen3-0.6B、bf16、MPS、本提示与评分协议、探索样本）。** 当前读取预算下，B/C/D0 的成绩很差，且错误多为复述问题中的编号。我们已经区分了三件事：直接选中哪些位置、资料可能怎样影响深层状态、最终回答是否正确。现有覆盖指标没有证明检索的必要或充分条件，也没有给训练索引器提供保证。

T2 在当前提示、生成和评分协议下未过基线门槛，没有进入稀疏对照。PR #6–#9 是同一批案例的后续诊断，不进入本课冻结主表，也不作为独立验证集。`03_G2_TO_G5.md` 保持 BLOCKED；本轮止于 PR #5 收尾与复审，不启动 G3、性能优化或新任务变式。